# Watch a clip and replay the model's recorded activity

This notebook runs the **Sun 2025 motion experiment** and lets you replay a selected stimulus alongside the model activity recorded for it. The model also reports motion direction and speed.

- **Run mode:** a remote GPU, prepared Sun data, and the cached example checkpoint are required. The default selection is four clips: one motion type in four flipped orientations.
- **Replay mode:** open a saved UMI experiment with the matching prepared data. No model is loaded and no inference runs.
- **Data:** registered and packaged under [brainscore/data/sun2025](../../brainscore/data/sun2025/README.md). The notebook does not duplicate the download or packaging pipeline.

The scientific question is whether a model's motion judgments agree with human judgments. Replay is an inspection tool for that experiment; it does not by itself test a causal hypothesis.

## 1. Configure the run

Run this notebook on the compute worker. Set `DATA_ROOT` to the output of the data plugin's preparation command. Set `MODE = 'replay'` and `SAVED_RUN` to an existing **model experiment directory** (the directory containing `experiment.json`) to inspect a saved run.

For a new run, `MODEL_ID = None` uses the supplied Qwen2.5-VL-3B native-video example. Replace it with a registered model identifier to use another subject, and choose one of its declared neural regions. That subject must support native clips, written answers and time-resolved activity for synchronized replay.

The example's V1 mapping is an illustrative layer selection, not a claim that its units are biological V1 neurons.

In [ ]:
import os
from pathlib import Path
from uuid import uuid4
from IPython.display import display

MODE = os.environ.get('SUN2025_NOTEBOOK_MODE', 'run')  # 'run' or 'replay'
DATA_ROOT = Path(os.environ.get('BRAINSCORE_SUN2025_ROOT', '~/.brainio/sun2025')).expanduser()
SAVED_RUN = Path(os.environ.get('SUN2025_RUN', './runs/sun2025/model')).expanduser()
NEW_RUN = Path(os.environ.get('SUN2025_NOTEBOOK_OUTPUT', f'./runs/sun2025-notebook-{uuid4().hex[:8]}'))
MODEL_ID = None  # e.g. a registered identifier with the required declared abilities
NEURAL_REGION = 'V1'
SCENES = [1]
MODULATIONS = [1]  # use list(range(1, 8)) for all seven motion types

if MODE not in {'run', 'replay'}:
    raise ValueError("MODE must be 'run' or 'replay'")
print({'mode': MODE, 'data': str(DATA_ROOT), 'scenes': SCENES, 'modulations': MODULATIONS})

## 2. Load the registered stimuli and design the task

A trial is one short clip followed by a direction-and-speed report. Keep all four flips for every selected scene and motion type: the human reference averages the four unflipped vectors. One protocol repetition creates one session and owns its resets.

In [ ]:
import brainscore
from brainscore.experiments import Experiment, ExperimentResult, RecordBehavior, RecordNeural
from brainscore.experiments.sun2025 import make_protocol, analyze

all_stimuli = brainscore.load_stimulus_set('Sun2025-second-order-motion', root=DATA_ROOT)
selected = all_stimuli.scene.isin(SCENES) & all_stimuli.modulation.isin(MODULATIONS)
stimuli = all_stimuli.loc[selected].copy().reset_index(drop=True)
if stimuli.empty:
    raise ValueError('None of the selected scene/modulation pairs is available in the prepared data')
expected_pairs = {(scene, modulation) for scene in SCENES for modulation in MODULATIONS}
actual_pairs = set(zip(stimuli.scene, stimuli.modulation))
if actual_pairs != expected_pairs:
    raise ValueError(f'Selected pairs missing from the prepared data: {expected_pairs - actual_pairs}')
stimuli.identifier = f'{all_stimuli.identifier}-scenes-{SCENES}-modulations-{MODULATIONS}'
stimuli.stimulus_paths = dict(zip(stimuli.stimulus_id, stimuli.video_path))
protocol = make_protocol(stimuli, seed=0)
print(f'{len(stimuli)} clip trials; inputs: {sorted(protocol.input_channels)}')
display(stimuli[['stimulus_id', 'scene', 'modulation', 'flip']])

## 3. Choose the subject and attach the instruments

Both recorders are visible here. The task asks for an answer; the instruments request saved behavior and neural activity from the **same presentation**. Replay mode skips model loading.

In [ ]:
if MODE == 'run':
    if MODEL_ID is None:
        import sys
        repo = Path(brainscore.__file__).resolve().parent.parent
        example = repo / 'examples' / 'sun2025'
        if not (example / 'qwen_video.py').is_file():
            raise FileNotFoundError('Open this notebook from the source checkout with examples/sun2025 available')
        sys.path.insert(0, str(example))
        from qwen_video import build_subject
        subject, provenance = build_subject()
    else:
        subject = brainscore.load_model(MODEL_ID)
        provenance = {'registered_model': MODEL_ID}
    print({'subject': subject.identifier,
           'inputs': sorted(subject.in_channels),
           'outputs': sorted(subject.out_channels)})
    recorders = [RecordBehavior(), RecordNeural([NEURAL_REGION])]
    experiment = Experiment(subject=subject, protocol=protocol, tools=recorders,
                            output_dir=NEW_RUN / 'model',
                            metadata={'execution_provenance': provenance})
    experiment.validate()
    print('Validated. The next cell runs the experiment.')
else:
    if not (SAVED_RUN / 'experiment.json').is_file():
        raise FileNotFoundError(f'No saved experiment at {SAVED_RUN}')
    print(f'Replaying the existing recording at {SAVED_RUN}')

## 4. Run once, or open the saved recording

The example uses the model's native video processor with all 16 frames. Its recording has eight temporal patches, each covering two frames. Other subjects may have different temporal resolution, which the replay reads from their recording coordinates.

In [ ]:
if MODE == 'run':
    result = experiment.run()
else:
    result = ExperimentResult(SAVED_RUN, None)

neural = result.neural(NEURAL_REGION)
behavior = result.behavior(measure='text')
print({'saved_at': str(result.directory), 'neural_dimensions': dict(neural.sizes),
       'behavior_dimensions': dict(behavior.sizes)})
print('Recording description:', neural.attrs.get('recording', {}))

## 5. Inspect the model's answers and compare with human reports

Selections are aligned by stimulus identity, not table order. The comparison preserves the raw answer, its parsed values and response format. It averages the four unflipped model vectors before comparing them with each of the three participants.

In [ ]:
import numpy as np

selected_ids = stimuli.stimulus_id.astype(str).tolist()
def select_presentations(assembly, identifiers):
    ids = list(assembly['stimulus_id'].values.astype(str))
    if len(ids) != len(set(ids)):
        raise ValueError('The recording contains duplicate stimulus identities')
    positions = {sid: index for index, sid in enumerate(ids)}
    missing = set(identifiers) - set(positions)
    if missing:
        raise ValueError(f'The saved recording is missing these selected trials: {sorted(missing)}')
    return assembly.isel(presentation=[positions[sid] for sid in identifiers])

neural = select_presentations(neural, selected_ids)
behavior = select_presentations(behavior, selected_ids)
human = brainscore.load_dataset('Sun2025-second-order-motion', root=DATA_ROOT)
human = select_presentations(human, sorted(stimuli.base_id.unique()))
trials, comparison, summary = analyze(behavior, stimuli, human)
summary['full_study'] = bool(summary['full_study'] and len(stimuli) == 1120)
display(trials[['stimulus_id', 'raw_answer', 'valid', 'response_format', 'parse_error']])
display(comparison)
summary

## 6. Replay one stimulus with its recorded activity

Choose a `STIMULUS_ID` from the table above and rerun the following cells. Playback has play/pause, frame stepping, a time slider, looping and speed controls. The heatmap cursor follows the matching recorded bin; values are held within that bin, with no interpolation.

The heatmap shows up to 64 evenly sampled units, normalized across this clip **for display only**. Hover a cell to see its raw value. The trace below it shows the raw mean absolute activity across all recorded units.

**Timing interpretation:** video time is not model computation time. A model can use the whole clip when computing a temporal patch. Playback does not imply causal processing order, neural latency, or a measured biological brain response.

In [ ]:
STIMULUS_ID = stimuli.stimulus_id.iloc[0]  # replace with any selected stimulus_id
from brainscore.experiments.sun2025.replay import replay_html
viewer_html = replay_html(stimuli, neural, STIMULUS_ID, behavior=behavior, display_units=64)
print(f'Replay prepared for {STIMULUS_ID}; no model inference was run by this cell.')

In [ ]:
import html
from IPython.display import HTML, display

display(HTML(
    '<iframe title="Stimulus and activity replay" sandbox="allow-scripts" '
    'style="width:100%;height:920px;border:0" '
    f'srcdoc="{html.escape(viewer_html, quote=True)}"></iframe>'
))

## 7. Export the replay

The HTML contains the selected clip frames, displayed activity and recorded answer. It opens independently of Jupyter and makes no network requests. This is a replay artifact; it does not overwrite recordings or replace any activity.

In [ ]:
import json
export_root = NEW_RUN if MODE == 'run' else SAVED_RUN.parent / f'replay-{uuid4().hex[:8]}'
export_root.mkdir(parents=True, exist_ok=True)
export_path = export_root / f'{STIMULUS_ID}-activity-replay.html'
export_path.write_text(viewer_html, encoding='utf-8')
receipt = {'mode': MODE, 'stimulus_id': STIMULUS_ID, 'trial_count': len(stimuli),
           'neural_shape': list(neural.shape), 'valid_motion_reports': summary['valid_answers'],
           'source_run': str(result.directory), 'replay_html': str(export_path)}
(export_root / 'notebook-summary.json').write_text(json.dumps(receipt, indent=2) + '\n')
print(f'Saved interactive replay: {export_path}')
receipt

## Conclusion and limits

The notebook makes the model's input, answer and measured activity inspectable together. A full experiment uses 40 scenes × seven motion types × four flips = 1,120 trials; the default four-trial selection is a demonstration.

The clip display and numerical report adapt the original human task. The study's open reference data is behavioral, so the activity replay is not a neural alignment score. It also does not establish that a displayed unit caused the answer; that would require a separately defined intervention experiment.

Source acquisition, packaging and reference-data registration remain in the [data plugin](../../brainscore/data/sun2025/README.md). The [experiment guide](../../examples/sun2025/README.md) describes the study adaptations and prior pilot evidence.